# YOLO Segmentation + MLflow (Multi-model)

This notebook is focused only on MLflow-managed training for:
- `yolo26n-seg.pt`
- `yolo26s-seg.pt`
- `yolo26m-seg.pt`
- `yolo26l-seg.pt`
- `yolo26x-seg.pt`

It expects a prepared dataset YAML (`data.yaml`) from your main pipeline.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import mlflow
from ultralytics import YOLO

import torch
torch.cuda.empty_cache()

# Set your data YAML path explicitly if needed
DATA_YAML = Path('cropped_736_aug_leaky/dataset/data.yaml')

# Fallback locations
if not DATA_YAML.exists():
    candidates = [
        Path('C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml'),
    ]
    for c in candidates:
        if c.exists():
            DATA_YAML = c
            break

if not DATA_YAML.exists():
    # Last-resort search (first match)
    found = sorted(Path('.').rglob('cropped_736_aug_leaky/dataset/data.yaml'))
    if found:
        DATA_YAML = found[0]

if not DATA_YAML.exists():
    raise FileNotFoundError('data.yaml not found. Prepare dataset first and set DATA_YAML manually.')

print('DATA_YAML:', DATA_YAML.resolve())


In [ ]:
# MLflow: train multiple YOLO-seg models (n/s/m/l/x) and log metrics/artifacts
from pathlib import Path
import json
import os
import sys
import threading

import cv2
import mlflow
from mlflow.exceptions import MlflowException
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO, settings as yolo_settings

# Ensure UTF-8 output to avoid MLflow unicode print errors on Windows cp1251 consoles
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

# If you have MLflow server, set e.g. "http://127.0.0.1:5000".
# If None, local file-backed MLflow store is used.
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"

MLFLOW_EXPERIMENT = "colony_yolo_seg_736_4"
LOCAL_MLFLOW_DIR = Path("mlruns_yolo_seg")

MODELS_TO_TRAIN = [
    # "yolo26n-seg.pt",
    # "yolo26s-seg.pt",
    "yolo26m-seg.pt",
    "yolo26l-seg.pt",
    "yolo26x-seg.pt",
]

MLFLOW_PROJECT = "runs/segment/runs/colony_seg_mlflow_736"
RUN_SUFFIX = "cropped736_offline_aug"
RESULTS_CSV_NAME = "results_736_4.csv"

# Training setup
TRAIN_EPOCHS = 50
TRAIN_BATCH = 8
TRAIN_IMGSZ = 736
TRAIN_PATIENCE = 80
TRAIN_DEVICE = None  # e.g. 0 for first GPU, or None for auto
STOP_ON_ERROR = False

# Custom metric inference setup (for Dice + counting quality)
PRED_CONF = 0.25
PRED_IOU = 0.7
PRED_MAX_DET = 300
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
LIVE_LOG_POLL_SEC = 2.0


def to_float(v):
    try:
        return float(v)
    except Exception:
        return float("nan")


def summarize_section(section, prefix):
    if section is None:
        return {
            f"precision_{prefix}": float("nan"),
            f"recall_{prefix}": float("nan"),
            f"mAP50_{prefix}": float("nan"),
            f"mAP50_95_{prefix}": float("nan"),
        }
    return {
        f"precision_{prefix}": to_float(getattr(section, "mp", float("nan"))),
        f"recall_{prefix}": to_float(getattr(section, "mr", float("nan"))),
        f"mAP50_{prefix}": to_float(getattr(section, "map50", float("nan"))),
        f"mAP50_95_{prefix}": to_float(getattr(section, "map", float("nan"))),
    }


def safe_metric_name(name: str) -> str:
    s = str(name).strip().replace(" ", "_")
    for bad in ["(", ")", "/", "\\", ":", ",", "|", "-", "."]:
        s = s.replace(bad, "_")
    while "__" in s:
        s = s.replace("__", "_")
    return s.strip("_")


def maybe_log_artifact(path: Path, artifact_path: str):
    if path.exists():
        mlflow.log_artifact(str(path), artifact_path=artifact_path)


def resolve_test_split_dirs(data_yaml_path: Path):
    data = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8")) or {}
    test_raw = data.get("test")
    if not test_raw:
        raise KeyError("`test` path is missing in data.yaml")

    test_img_dir = Path(str(test_raw).strip().strip('"').strip("'"))
    if not test_img_dir.is_absolute():
        test_img_dir = (data_yaml_path.parent / test_img_dir).resolve()

    split_name = test_img_dir.name
    label_candidates = [data_yaml_path.parent / "labels" / split_name]

    if test_img_dir.parent.name == "images":
        label_candidates.append(test_img_dir.parent.parent / "labels" / split_name)

    replaced = Path(
        str(test_img_dir)
        .replace("\\images\\", "\\labels\\")
        .replace("/images/", "/labels/")
    )
    label_candidates.append(replaced)

    for cand in label_candidates:
        if cand.exists():
            return test_img_dir, cand

    return test_img_dir, label_candidates[0]


def read_yolo_seg_polygons(label_path: Path):
    if not label_path.exists():
        return []

    txt = label_path.read_text(encoding="utf-8", errors="ignore")
    if not txt.strip():
        return []
    # Fix malformed files containing literal "\n" between coordinates.
    txt = txt.replace("\r", "").replace("\n", chr(10))

    polygons = []
    for raw_line in txt.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 7:
            continue

        coords = []
        for token in parts[1:]:
            try:
                coords.append(float(token))
            except Exception:
                for sub in token.replace(",", " ").replace(";", " ").split():
                    try:
                        coords.append(float(sub))
                    except Exception:
                        pass

        if len(coords) < 6:
            continue
        if len(coords) % 2 == 1:
            coords = coords[:-1]

        pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
        pts = np.clip(pts, 0.0, 1.0)
        if pts.shape[0] >= 3:
            polygons.append(pts)

    return polygons


def polygons_norm_to_mask(polygons_norm, h: int, w: int):
    mask = np.zeros((h, w), dtype=np.uint8)
    if h <= 0 or w <= 0:
        return mask

    for pts in polygons_norm:
        arr = np.asarray(pts, dtype=np.float32)
        if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
            continue

        arr_px = np.empty_like(arr)
        arr_px[:, 0] = np.clip(arr[:, 0] * (w - 1), 0, w - 1)
        arr_px[:, 1] = np.clip(arr[:, 1] * (h - 1), 0, h - 1)
        cv2.fillPoly(mask, [np.round(arr_px).astype(np.int32)], 1)

    return mask


def result_to_pred_mask(result, h: int, w: int):
    mask = np.zeros((h, w), dtype=np.uint8)
    pred_count = 0

    masks_obj = getattr(result, "masks", None)
    if masks_obj is None:
        return mask, pred_count

    polys = getattr(masks_obj, "xy", None)
    if polys is not None and len(polys) > 0:
        pred_count = int(len(polys))
        for poly in polys:
            arr = np.asarray(poly, dtype=np.float32)
            if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
                continue
            arr[:, 0] = np.clip(arr[:, 0], 0, w - 1)
            arr[:, 1] = np.clip(arr[:, 1], 0, h - 1)
            cv2.fillPoly(mask, [np.round(arr).astype(np.int32)], 1)
        return mask, pred_count

    data = getattr(masks_obj, "data", None)
    if data is None:
        return mask, pred_count

    arr = data.detach().cpu().numpy()
    pred_count = int(arr.shape[0])
    if arr.size == 0:
        return mask, pred_count

    union = (arr > 0.5).any(axis=0).astype(np.uint8)
    if union.shape != (h, w):
        union = cv2.resize(union, (w, h), interpolation=cv2.INTER_NEAREST)
    return union.astype(np.uint8), pred_count


def dice_score(pred_mask, gt_mask, eps: float = 1e-7):
    a = pred_mask.astype(bool)
    b = gt_mask.astype(bool)

    sa = float(a.sum(dtype=np.float64))
    sb = float(b.sum(dtype=np.float64))
    if sa == 0.0 and sb == 0.0:
        return 1.0

    inter = float(np.logical_and(a, b).sum(dtype=np.float64))
    return float((2.0 * inter + eps) / (sa + sb + eps))


def find_experiment_any_state(client: MlflowClient, experiment_name: str):
    for exp in client.search_experiments(view_type=ViewType.ALL):
        if exp.name == experiment_name:
            return exp
    return None


def set_or_restore_experiment(experiment_name: str):
    client = MlflowClient()
    exp = find_experiment_any_state(client, experiment_name)

    if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
        print(f"Experiment '{experiment_name}' is deleted. Restoring...")
        client.restore_experiment(exp.experiment_id)

    try:
        return mlflow.set_experiment(experiment_name)
    except MlflowException as e:
        msg = str(e).lower()
        if "deleted experiment" in msg:
            exp = find_experiment_any_state(client, experiment_name)
            if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
                client.restore_experiment(exp.experiment_id)
                return mlflow.set_experiment(experiment_name)
        raise


def close_active_mlflow_runs(max_iters: int = 10):
    """Best-effort cleanup of dangling active MLflow runs."""
    closed = 0
    for _ in range(max_iters):
        active = mlflow.active_run()
        if active is None:
            break
        try:
            mlflow.end_run()
            closed += 1
        except Exception as e:
            print(f"[WARN] Failed to end active MLflow run: {e}")
            break
    return closed


def resolve_checkpoint_for_training(ckpt_name: str) -> str:
    ckpt_path = Path(ckpt_name)
    if not ckpt_path.exists():
        # Let Ultralytics download by model name.
        return ckpt_name

    try:
        # Validate checkpoint with standard torch load.
        _ = torch.load(str(ckpt_path), map_location="cpu")
        return str(ckpt_path)
    except Exception:
        broken = ckpt_path.with_name(f"{ckpt_path.name}.broken")
        suffix = 1
        while broken.exists():
            broken = ckpt_path.with_name(f"{ckpt_path.name}.broken{suffix}")
            suffix += 1
        ckpt_path.rename(broken)
        print(f"[WARN] Corrupt checkpoint moved to: {broken}")
        print(f"[WARN] Will try to auto-download fresh weights for: {ckpt_name}")
        return ckpt_name


def compute_test_custom_metrics(best_model, data_yaml: Path, imgsz: int, device):
    test_img_dir, test_lbl_dir = resolve_test_split_dirs(data_yaml)
    if not test_img_dir.exists():
        raise FileNotFoundError(f"Test image dir not found: {test_img_dir}")

    test_images = [
        p for p in sorted(test_img_dir.iterdir())
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    ]
    if not test_images:
        empty_metrics = {
            "dice_M_mean": float("nan"),
            "dice_M_median": float("nan"),
            "mae_count": float("nan"),
            "rmse_count": float("nan"),
            "mape_count_nonzero": float("nan"),
            "test_images_eval": 0,
            "test_images_missing_labels": 0,
        }
        return empty_metrics, pd.DataFrame()

    pred_iter = best_model.predict(
        source=str(test_img_dir),
        imgsz=imgsz,
        conf=PRED_CONF,
        iou=PRED_IOU,
        max_det=PRED_MAX_DET,
        device=device,
        stream=True,
        verbose=False,
        save=False,
    )

    rows = []
    for res in pred_iter:
        image_path = Path(res.path)
        h, w = map(int, res.orig_shape)

        label_path = test_lbl_dir / f"{image_path.stem}.txt"
        gt_polys = read_yolo_seg_polygons(label_path)
        gt_mask = polygons_norm_to_mask(gt_polys, h, w)

        pred_mask, pred_count = result_to_pred_mask(res, h, w)
        gt_count = int(len(gt_polys))
        count_error = int(pred_count - gt_count)
        abs_error = abs(count_error)
        ape = (abs_error / gt_count) if gt_count > 0 else float("nan")

        rows.append(
            {
                "image": image_path.name,
                "label_exists": int(label_path.exists()),
                "gt_count": gt_count,
                "pred_count": int(pred_count),
                "count_error": count_error,
                "count_abs_error": abs_error,
                "count_ape": float(ape),
                "dice": dice_score(pred_mask, gt_mask),
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        metrics = {
            "dice_M_mean": float("nan"),
            "dice_M_median": float("nan"),
            "mae_count": float("nan"),
            "rmse_count": float("nan"),
            "mape_count_nonzero": float("nan"),
            "test_images_eval": 0,
            "test_images_missing_labels": 0,
        }
        return metrics, df

    sq = np.square(df["count_error"].to_numpy(dtype=np.float64))
    ape_valid = df["count_ape"].dropna()

    metrics = {
        "dice_M_mean": float(df["dice"].mean()),
        "dice_M_median": float(df["dice"].median()),
        "mae_count": float(df["count_abs_error"].mean()),
        "rmse_count": float(np.sqrt(sq.mean())),
        "mape_count_nonzero": float(ape_valid.mean() * 100.0) if len(ape_valid) else float("nan"),
        "test_images_eval": int(len(df)),
        "test_images_missing_labels": int((df["label_exists"] == 0).sum()),
    }
    return metrics, df


def find_train_results_csv(train_dir: Path, project_dir: str, run_name: str):
    candidates = []
    names = [RESULTS_CSV_NAME, "results.csv"]

    if train_dir is not None:
        for nm in names:
            candidates.append(train_dir / nm)

    proj = Path(project_dir)
    for nm in names:
        candidates.append(proj / run_name / nm)
        candidates.append(Path("runs") / "segment" / proj / run_name / nm)

    for base in [proj, Path("runs") / "segment" / proj, Path("runs") / "segment"]:
        if base.exists():
            try:
                for nm in names:
                    candidates.extend(
                        sorted(
                            base.glob(f"**/{run_name}*/{nm}"),
                            key=lambda p: p.stat().st_mtime,
                            reverse=True,
                        )
                    )
            except Exception:
                pass

    for c in candidates:
        if c is not None and c.exists():
            return c
    return None

def pick_active_results_csv(results_csv_candidates):
    existing = [p for p in results_csv_candidates if p is not None and p.exists()]
    if not existing:
        return None
    try:
        return max(existing, key=lambda p: p.stat().st_mtime)
    except Exception:
        return existing[0]


def log_partial_train_metrics_to_run(run_id: str, results_csv: Path):
    if not run_id or results_csv is None or not results_csv.exists():
        return False

    try:
        df = pd.read_csv(results_csv)
    except Exception as e:
        print(f"[WARN] Cannot read partial results.csv: {e}")
        return False

    if df.empty:
        return False

    client = MlflowClient()
    logged_any = False
    for step_idx, row in df.iterrows():
        step_raw = row.get("epoch", step_idx)
        step_float = to_float(step_raw)
        step = int(step_float) if np.isfinite(step_float) else int(step_idx)

        for k, v in row.items():
            vv = to_float(v)
            if np.isfinite(vv):
                client.log_metric(run_id, f"train_{safe_metric_name(k)}", float(vv), step=step)
                logged_any = True

    return logged_any


def stream_train_metrics_live(run_id: str, results_csv_candidates, stop_event, poll_seconds: float = LIVE_LOG_POLL_SEC):
    """Stream new rows from Ultralytics results.csv to MLflow as train_* metrics."""
    client = MlflowClient()
    next_row = 0
    active_csv = None

    while not stop_event.is_set():
        current_csv = pick_active_results_csv(results_csv_candidates)

        if current_csv is not None and current_csv != active_csv:
            active_csv = current_csv
            next_row = 0
            print(f"[LIVE] train metrics source: {active_csv}")

        if active_csv is not None and active_csv.exists():
            try:
                df = pd.read_csv(active_csv)
            except Exception:
                if stop_event.wait(poll_seconds):
                    break
                continue

            if len(df) > next_row:
                for row_idx in range(next_row, len(df)):
                    row = df.iloc[row_idx]
                    step_raw = row.get("epoch", row_idx)
                    step_float = to_float(step_raw)
                    step = int(step_float) if np.isfinite(step_float) else int(row_idx)

                    for k, v in row.items():
                        vv = to_float(v)
                        if np.isfinite(vv):
                            client.log_metric(run_id, f"train_{safe_metric_name(k)}", float(vv), step=step)
                next_row = len(df)

        if stop_event.wait(poll_seconds):
            break

    # Final flush in case the last rows were written right before stop.
    if active_csv is not None and active_csv.exists():
        try:
            df = pd.read_csv(active_csv)
            if len(df) > next_row:
                for row_idx in range(next_row, len(df)):
                    row = df.iloc[row_idx]
                    step_raw = row.get("epoch", row_idx)
                    step_float = to_float(step_raw)
                    step = int(step_float) if np.isfinite(step_float) else int(row_idx)

                    for k, v in row.items():
                        vv = to_float(v)
                        if np.isfinite(vv):
                            client.log_metric(run_id, f"train_{safe_metric_name(k)}", float(vv), step=step)
        except Exception:
            pass


# Make this cell self-contained: resolve DATA_YAML if previous setup cell was not executed.
if "DATA_YAML" not in globals() or DATA_YAML is None:
    DATA_YAML = Path("cropped_736_aug_leaky/dataset/data.yaml")
else:
    DATA_YAML = Path(DATA_YAML)

if not DATA_YAML.exists():
    fallback_candidates = [
        Path("C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml"),
        Path("cropped_736/dataset/data.yaml"),
        Path("C:/ColonyNet/cropped_736/dataset/data.yaml"),
    ]
    for candidate in fallback_candidates:
        if candidate.exists():
            DATA_YAML = candidate
            break

if not DATA_YAML.exists():
    found = sorted(Path(".").rglob("cropped_736_aug_leaky/dataset/data.yaml"))
    if not found:
        found = sorted(Path(".").rglob("cropped_736/dataset/data.yaml"))
    if found:
        DATA_YAML = found[0]

if not DATA_YAML.exists():
    raise FileNotFoundError("data.yaml not found. Run dataset preparation first or set DATA_YAML manually.")

print("Using DATA_YAML:", DATA_YAML.resolve())

# Enable Ultralytics MLflow autologging (alongside manual custom logging below).
AUTOLOG_ULTRALYTICS = True
yolo_settings.update({"mlflow": AUTOLOG_ULTRALYTICS})
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT
print("Ultralytics setting mlflow:", yolo_settings.get("mlflow"))

# Configure tracking
if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
else:
    LOCAL_MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(LOCAL_MLFLOW_DIR.resolve().as_uri())

active_experiment = set_or_restore_experiment(MLFLOW_EXPERIMENT)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", getattr(active_experiment, "name", MLFLOW_EXPERIMENT))
print("Experiment ID:", getattr(active_experiment, "experiment_id", "unknown"))

summary_rows = []

for ckpt in MODELS_TO_TRAIN:
    model_tag = Path(ckpt).stem
    run_name = f"{model_tag}_{RUN_SUFFIX}"
    print(f"\n=== Training {ckpt} ===")

    run_id = None
    train_dir = None

    closed_before = close_active_mlflow_runs()
    if closed_before:
        print(f"[INFO] Closed {closed_before} stale active MLflow run(s) before {ckpt}")

    try:
        with mlflow.start_run(run_name=run_name) as active_run:
            run_id = active_run.info.run_id
            train_ckpt = resolve_checkpoint_for_training(ckpt)

            mlflow.set_tags(
                {
                    "framework": "ultralytics",
                    "task": "segment",
                    "dataset": str(DATA_YAML),
                    "model_ckpt": ckpt,
                    "offline_aug": "true",
                    "online_aug": "false",
                }
            )
            mlflow.log_metric("run_started", 1.0, step=0)
            mlflow.log_params(
                {
                    "model": ckpt,
                    "model_effective": train_ckpt,
                    "data_yaml": str(DATA_YAML),
                    "imgsz": TRAIN_IMGSZ,
                    "epochs": TRAIN_EPOCHS,
                    "batch": TRAIN_BATCH,
                    "patience": TRAIN_PATIENCE,
                    "project": MLFLOW_PROJECT,
                    "run_name": run_name,
                    "tracking_uri": mlflow.get_tracking_uri(),
                    "predict_conf": PRED_CONF,
                    "predict_iou": PRED_IOU,
                    "predict_max_det": PRED_MAX_DET,
                }
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                print("CUDA cache cleared")

            expected_train_dir = Path(MLFLOW_PROJECT) / run_name
            live_results_candidates = [
                expected_train_dir / "results.csv",
                Path("runs") / "segment" / expected_train_dir / "results.csv",
                Path("runs") / "segment" / run_name / "results.csv",
            ]
            live_stop_event = threading.Event()
            live_thread = threading.Thread(
                target=stream_train_metrics_live,
                args=(run_id, live_results_candidates, live_stop_event),
                kwargs={"poll_seconds": LIVE_LOG_POLL_SEC},
                daemon=True,
            )
            live_thread.start()

            model = YOLO(train_ckpt)
            try:
                train_results = model.train(
                    data=str(DATA_YAML),
                    task="segment",
                    imgsz=TRAIN_IMGSZ,
                    epochs=TRAIN_EPOCHS,
                    batch=TRAIN_BATCH,
                    patience=TRAIN_PATIENCE,
                    device=TRAIN_DEVICE,
                    project=MLFLOW_PROJECT,
                    name=run_name,
                    exist_ok=True,
                    plots=True,
                    # No online augmentation (offline-augmented dataset only)
                    degrees=0.0,
                    translate=0.0,
                    scale=0.0,
                    shear=0.0,
                    perspective=0.0,
                    fliplr=0.0,
                    flipud=0.0,
                    hsv_h=0.0,
                    hsv_s=0.0,
                    hsv_v=0.0,
                    mosaic=0.0,
                    mixup=0.0,
                    copy_paste=0.0,
                    erasing=0.0,
                )
            finally:
                live_stop_event.set()
                live_thread.join(timeout=15)

            train_dir = Path(train_results.save_dir)
            raw_results_csv = train_dir / "results.csv"
            results_csv = train_dir / RESULTS_CSV_NAME
            if raw_results_csv.exists():
                try:
                    results_csv.write_bytes(raw_results_csv.read_bytes())
                except Exception:
                    results_csv = raw_results_csv
            elif not results_csv.exists():
                results_csv = raw_results_csv

            best_w = train_dir / "weights" / "best.pt"
            last_w = train_dir / "weights" / "last.pt"

            # Log train artifacts
            maybe_log_artifact(train_dir / "args.yaml", "train")
            maybe_log_artifact(results_csv, "train")
            maybe_log_artifact(train_dir / "results.png", "train")
            maybe_log_artifact(train_dir / "confusion_matrix.png", "train")
            maybe_log_artifact(train_dir / "confusion_matrix_normalized.png", "train")
            for nm in [
                "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png", "BoxF1_curve.png",
                "MaskPR_curve.png", "MaskP_curve.png", "MaskR_curve.png", "MaskF1_curve.png",
            ]:
                maybe_log_artifact(train_dir / nm, "train")
            maybe_log_artifact(best_w, "weights")
            maybe_log_artifact(last_w, "weights")

            # Log last epoch train metrics from results_736_4.csv (fallback results.csv)
            r_csv = results_csv
            if r_csv.exists():
                train_df = pd.read_csv(r_csv)
                if len(train_df) > 0:
                    last_row = train_df.iloc[-1].to_dict()
                    train_metrics = {}
                    for k, v in last_row.items():
                        vv = to_float(v)
                        if np.isfinite(vv):
                            train_metrics[f"train_{safe_metric_name(k)}"] = vv
                    if train_metrics:
                        mlflow.log_metrics(train_metrics)

            if not best_w.exists():
                raise FileNotFoundError(f"best.pt not found for {ckpt}: {best_w}")

            # Evaluate on test split
            best_model = YOLO(str(best_w))
            test_results = best_model.val(
                data=str(DATA_YAML),
                split="test",
                imgsz=TRAIN_IMGSZ,
                project=MLFLOW_PROJECT,
                name=f"{run_name}_test",
                exist_ok=True,
                plots=True,
                save_json=True,
            )

            test_dir = Path(test_results.save_dir)

            metrics_summary = {}
            metrics_summary.update(summarize_section(getattr(test_results, "box", None), "B"))
            metrics_summary.update(summarize_section(getattr(test_results, "seg", None), "M"))
            metrics_summary["fitness"] = to_float(getattr(test_results, "fitness", float("nan")))

            # Custom post-hoc metrics on test split
            custom_metrics, per_image_df = compute_test_custom_metrics(
                best_model=best_model,
                data_yaml=Path(DATA_YAML),
                imgsz=TRAIN_IMGSZ,
                device=TRAIN_DEVICE,
            )
            metrics_summary.update(custom_metrics)

            per_image_csv = test_dir / "test_per_image_metrics.csv"
            per_image_df.to_csv(per_image_csv, index=False)

            # Persist + log test metrics json
            test_json = test_dir / "test_metrics_summary.json"
            with test_json.open("w", encoding="utf-8") as f:
                json.dump(metrics_summary, f, indent=2)

            mlflow_metrics = {k: v for k, v in metrics_summary.items() if np.isfinite(v)}
            if mlflow_metrics:
                mlflow.log_metrics(mlflow_metrics)

            # Log test artifacts
            maybe_log_artifact(test_json, "test")
            maybe_log_artifact(per_image_csv, "test")
            maybe_log_artifact(test_dir / "predictions.json", "test")
            maybe_log_artifact(test_dir / "confusion_matrix.png", "test")
            maybe_log_artifact(test_dir / "confusion_matrix_normalized.png", "test")
            for nm in [
                "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png", "BoxF1_curve.png",
                "MaskPR_curve.png", "MaskP_curve.png", "MaskR_curve.png", "MaskF1_curve.png",
                "PR_curve.png", "P_curve.png", "R_curve.png", "F1_curve.png",
            ]:
                maybe_log_artifact(test_dir / nm, "test")

            mlflow.log_params(
                {
                    "ultralytics_train_dir": str(train_dir.resolve()),
                    "ultralytics_test_dir": str(test_dir.resolve()),
                    "best_weights": str(best_w.resolve()),
                }
            )

            row = {
                "model": ckpt,
                "train_dir": str(train_dir),
                "test_dir": str(test_dir),
                **metrics_summary,
            }
            summary_rows.append(row)

            print(f"Done: {ckpt}")
            print("  train_dir:", train_dir)
            print("  test_dir :", test_dir)

            # Free references before next model
            del model
            del best_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    except Exception as exc:
        print(f"[ERROR] {ckpt}: {exc}")

        try:
            partial_csv = find_train_results_csv(train_dir, MLFLOW_PROJECT, run_name)
            if run_id:
                client = MlflowClient()
                client.set_tag(run_id, "error_message", str(exc)[:1000])
                if partial_csv is not None:
                    logged = log_partial_train_metrics_to_run(run_id, partial_csv)
                    if logged:
                        print(f"[INFO] Logged partial train metrics from: {partial_csv}")
                    else:
                        print(f"[INFO] Partial results found but no numeric metrics: {partial_csv}")
        except Exception as log_exc:
            print(f"[WARN] Failed to log partial metrics: {log_exc}")

        summary_rows.append({"model": ckpt, "error": str(exc)})
        if STOP_ON_ERROR:
            raise
    finally:
        closed_after = close_active_mlflow_runs()
        if closed_after:
            print(f"[INFO] Closed {closed_after} dangling MLflow run(s) after {ckpt}")

summary_df = pd.DataFrame(summary_rows)
try:
    display(summary_df)
except Exception:
    print(summary_df)

summary_csv = Path("mlflow_multi_model_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print("Saved summary:", summary_csv.resolve())


In [ ]:
# X-only: train yolo26x-seg.pt in same MLflow experiment/project
from pathlib import Path
from datetime import datetime
import os, sys
import numpy as np
import pandas as pd
import torch
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
from mlflow.exceptions import MlflowException
from ultralytics import YOLO, settings as yolo_settings

os.environ.setdefault("PYTHONIOENCODING", "utf-8")
try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "colony_yolo_seg"
MLFLOW_PROJECT = "runs/colony_seg_mlflow"

CKPT_X = "yolo26x-seg.pt"  # если файла нет, Ultralytics попытается скачать
RUN_NAME = f"yolo26x-seg_cropped736_offline_aug_xonly_{datetime.now():%Y%m%d_%H%M%S}"

TRAIN_EPOCHS = 50
TRAIN_BATCH = 8
TRAIN_IMGSZ = 736
TRAIN_PATIENCE = 80
TRAIN_DEVICE = None
RESULTS_CSV_NAME = "results_736_4.csv"

def to_float(v):
    try:
        return float(v)
    except Exception:
        return float("nan")

def safe_metric_name(name: str) -> str:
    s = str(name).strip().replace(" ", "_")
    for bad in ["(", ")", "/", "\\", ":", ",", "|", "-", "."]:
        s = s.replace(bad, "_")
    while "__" in s:
        s = s.replace("__", "_")
    return s.strip("_")

def summarize_section(section, prefix):
    if section is None:
        return {
            f"precision_{prefix}": float("nan"),
            f"recall_{prefix}": float("nan"),
            f"mAP50_{prefix}": float("nan"),
            f"mAP50_95_{prefix}": float("nan"),
        }
    return {
        f"precision_{prefix}": to_float(getattr(section, "mp", float("nan"))),
        f"recall_{prefix}": to_float(getattr(section, "mr", float("nan"))),
        f"mAP50_{prefix}": to_float(getattr(section, "map50", float("nan"))),
        f"mAP50_95_{prefix}": to_float(getattr(section, "map", float("nan"))),
    }

def find_experiment_any_state(client: MlflowClient, experiment_name: str):
    for exp in client.search_experiments(view_type=ViewType.ALL):
        if exp.name == experiment_name:
            return exp
    return None

def set_or_restore_experiment(experiment_name: str):
    client = MlflowClient()
    exp = find_experiment_any_state(client, experiment_name)
    if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
        client.restore_experiment(exp.experiment_id)
    try:
        return mlflow.set_experiment(experiment_name)
    except MlflowException as e:
        if "deleted experiment" in str(e).lower():
            exp = find_experiment_any_state(client, experiment_name)
            if exp is not None and getattr(exp, "lifecycle_stage", "active") != "active":
                client.restore_experiment(exp.experiment_id)
                return mlflow.set_experiment(experiment_name)
        raise


def close_active_mlflow_runs(max_iters: int = 10):
    """Best-effort cleanup of dangling active MLflow runs."""
    closed = 0
    for _ in range(max_iters):
        active = mlflow.active_run()
        if active is None:
            break
        try:
            mlflow.end_run()
            closed += 1
        except Exception as e:
            print(f"[WARN] Failed to end active MLflow run: {e}")
            break
    return closed


# DATA_YAML fallback
if "DATA_YAML" not in globals() or DATA_YAML is None:
    DATA_YAML = Path("cropped_736_aug_leaky/dataset/data.yaml")
else:
    DATA_YAML = Path(DATA_YAML)

if not DATA_YAML.exists():
    raise FileNotFoundError(f"DATA_YAML not found: {DATA_YAML}")

# enable built-in Ultralytics mlflow autologging
AUTOLOG_ULTRALYTICS = True
yolo_settings.update({"mlflow": AUTOLOG_ULTRALYTICS})
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
set_or_restore_experiment(MLFLOW_EXPERIMENT)

print("DATA_YAML:", DATA_YAML.resolve())
print("Run name :", RUN_NAME)

closed_before = close_active_mlflow_runs()
if closed_before:
    print(f"[INFO] Closed {closed_before} stale active MLflow run(s) before {RUN_NAME}")

with mlflow.start_run(run_name=RUN_NAME) as run:
    run_id = run.info.run_id
    mlflow.log_metric("run_started", 1.0, step=0)
    mlflow.log_params({
        "model": CKPT_X,
        "data_yaml": str(DATA_YAML),
        "imgsz": TRAIN_IMGSZ,
        "epochs": TRAIN_EPOCHS,
        "batch": TRAIN_BATCH,
        "patience": TRAIN_PATIENCE,
        "project": MLFLOW_PROJECT,
        "run_name": RUN_NAME,
        "tracking_uri": mlflow.get_tracking_uri(),
        "online_aug": "false",
    })
    mlflow.set_tags({
        "framework": "ultralytics",
        "task": "segment",
        "dataset": str(DATA_YAML),
        "model_ckpt": CKPT_X,
        "offline_aug": "true",
        "online_aug": "false",
    })

    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        model = YOLO(CKPT_X)
        train_results = model.train(
            data=str(DATA_YAML),
            task="segment",
            imgsz=TRAIN_IMGSZ,
            epochs=TRAIN_EPOCHS,
            batch=TRAIN_BATCH,
            patience=TRAIN_PATIENCE,
            device=TRAIN_DEVICE,
            project=MLFLOW_PROJECT,
            name=RUN_NAME,
            exist_ok=False,
            plots=True,
            degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
            fliplr=0.0, flipud=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
            mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0,
        )

        train_dir = Path(train_results.save_dir)
        raw_results_csv = train_dir / "results.csv"
        results_csv = train_dir / RESULTS_CSV_NAME
        if raw_results_csv.exists():
            try:
                results_csv.write_bytes(raw_results_csv.read_bytes())
            except Exception:
                results_csv = raw_results_csv
        elif not results_csv.exists():
            results_csv = raw_results_csv

        best_w = train_dir / "weights" / "best.pt"
        last_w = train_dir / "weights" / "last.pt"

        # online-like logging from csv (all epochs)
        if results_csv.exists():
            df = pd.read_csv(results_csv)
            for i, row in df.iterrows():
                step_raw = row.get("epoch", i)
                step = int(to_float(step_raw)) if np.isfinite(to_float(step_raw)) else int(i)
                for k, v in row.items():
                    vv = to_float(v)
                    if np.isfinite(vv):
                        mlflow.log_metric(f"train_{safe_metric_name(k)}", vv, step=step)
            mlflow.log_artifact(str(results_csv), artifact_path="train")

        # train artifacts
        for p in [
            train_dir / "args.yaml",
            train_dir / "results.png",
            train_dir / "confusion_matrix.png",
            train_dir / "confusion_matrix_normalized.png",
            best_w, last_w
        ]:
            if p.exists():
                mlflow.log_artifact(str(p), artifact_path="train" if p.suffix != ".pt" else "weights")

        if not best_w.exists():
            raise FileNotFoundError(f"best.pt not found: {best_w}")

        # test validation
        best_model = YOLO(str(best_w))
        test_results = best_model.val(
            data=str(DATA_YAML),
            split="test",
            imgsz=TRAIN_IMGSZ,
            project=MLFLOW_PROJECT,
            name=f"{RUN_NAME}_test",
            exist_ok=True,
            plots=True,
            save_json=True,
        )
        test_dir = Path(test_results.save_dir)

        metrics_summary = {}
        metrics_summary.update(summarize_section(getattr(test_results, "box", None), "B"))
        metrics_summary.update(summarize_section(getattr(test_results, "seg", None), "M"))
        metrics_summary["fitness"] = to_float(getattr(test_results, "fitness", float("nan")))

        mlflow.log_metrics({k: v for k, v in metrics_summary.items() if np.isfinite(v)})

        test_json = test_dir / "test_metrics_summary.json"
        import json
        with test_json.open("w", encoding="utf-8") as f:
            json.dump(metrics_summary, f, indent=2)

        for p in [
            test_json,
            test_dir / "predictions.json",
            test_dir / "confusion_matrix.png",
            test_dir / "confusion_matrix_normalized.png",
        ]:
            if p.exists():
                mlflow.log_artifact(str(p), artifact_path="test")

        mlflow.log_params({
            "ultralytics_train_dir": str(train_dir.resolve()),
            "ultralytics_test_dir": str(test_dir.resolve()),
            "best_weights": str(best_w.resolve()),
        })

        print("DONE:", RUN_NAME)
        print("train_dir:", train_dir)
        print("test_dir :", test_dir)

    except Exception as e:
        MlflowClient().set_tag(run_id, "error_message", str(e)[:1000])
        print("[ERROR]", e)
        raise

closed_after = close_active_mlflow_runs()
if closed_after:
    print(f"[INFO] Closed {closed_after} dangling MLflow run(s) after {RUN_NAME}")


In [ ]:
# Preview segmentation masks and contours only (no bounding boxes) for all trained YOLO models
from pathlib import Path
import re
import colorsys

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
PREVIEW_IMAGE_PATHS = [
    Path("cropped_736/images/train/1127763_IMG_4363.jpg"),
    Path("cropped_736/images/train/1127763_IMG_4615.jpg"),
    Path("cropped_736/images/train/1127763_IMG_4672.jpg"),
    Path("cropped_736/images/train/1127763_IMG_6137.jpg"),
]
PRED_CONF = 0.25
PRED_IOU = 0.60
MASK_ALPHA = 0.55
COLS = 2
CONTOUR_THICKNESS = 2
SMOOTH_SIGMA = 0.9
SAVE_PREVIEW_PNG = True
PREVIEW_SAVE_DIR = Path("runs/segment/runs/colony_seg_mlflow/preview_masks_only")


MODEL_ORDER = ["yolo26n-seg", "yolo26s-seg", "yolo26m-seg", "yolo26l-seg", "yolo26x-seg"]
MODEL_TAG_RE = re.compile(r"^(yolo26[a-z]-seg)", re.IGNORECASE)


if SAVE_PREVIEW_PNG:
    PREVIEW_SAVE_DIR.mkdir(parents=True, exist_ok=True)


def resolve_preview_images():
    selected = []
    missing = []

    for rel_path in PREVIEW_IMAGE_PATHS:
        p = Path(rel_path)
        if p.exists():
            selected.append(p)
            continue

        p_abs = Path("C:/ColonyNet") / p
        if p_abs.exists():
            selected.append(p_abs)
        else:
            missing.append(str(rel_path))

    if missing:
        raise FileNotFoundError("Missing preview images:" + "\n" + "\n".join(missing))

    return selected



def collect_trained_best_weights():
    by_model = {}

    # Include current globals if available
    extra_candidates = []
    if "best_weights" in globals():
        try:
            p = Path(best_weights)
            if p.exists():
                extra_candidates.append(p)
        except Exception:
            pass

    if "run_dir" in globals():
        try:
            p = Path(run_dir) / "weights" / "best.pt"
            if p.exists():
                extra_candidates.append(p)
        except Exception:
            pass

    for p in extra_candidates:
        run_name = p.parent.parent.name
        m = MODEL_TAG_RE.match(run_name)
        if not m:
            continue
        model_tag = m.group(1).lower()
        prev = by_model.get(model_tag)
        if prev is None or p.stat().st_mtime > prev[1].stat().st_mtime:
            by_model[model_tag] = (run_name, p)

    bases = [
        Path("runs/segment/runs/colony_seg_mlflow"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
    ]

    for base in bases:
        if not base.exists():
            continue
        for d in base.iterdir():
            if not d.is_dir() or d.name.endswith("_test"):
                continue
            best = d / "weights" / "best.pt"
            if not best.exists():
                continue

            m = MODEL_TAG_RE.match(d.name)
            if not m:
                continue
            model_tag = m.group(1).lower()

            prev = by_model.get(model_tag)
            if prev is None or best.stat().st_mtime > prev[1].stat().st_mtime:
                by_model[model_tag] = (d.name, best)

    if not by_model:
        raise FileNotFoundError("No trained best.pt found in colony_seg_mlflow runs.")

    ordered = []
    for tag in MODEL_ORDER:
        if tag in by_model:
            run_name, best = by_model[tag]
            ordered.append((tag, run_name, best))

    # Append any extra tags not in MODEL_ORDER
    for tag, (run_name, best) in sorted(by_model.items()):
        if tag not in MODEL_ORDER:
            ordered.append((tag, run_name, best))

    return ordered


def color_for_idx(i, n_total=1):
    # High-contrast deterministic instance colors.
    if n_total < 1:
        n_total = 1
    t = ((i * 37) % n_total) / max(1, n_total - 1)
    hue = (0.02 + 0.96 * t) % 1.0
    sat = 0.95
    val = 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
    return np.array([255.0 * r, 255.0 * g, 255.0 * b], dtype=np.float32)


def render_masks_only(image_path, result):
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        return None, 0

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    overlay = rgb.copy()
    h, w = overlay.shape[:2]

    n_masks = 0
    if result.masks is not None and getattr(result.masks, "data", None) is not None:
        mask_data = result.masks.data
        if hasattr(mask_data, "detach"):
            mask_stack = mask_data.detach().cpu().numpy()
        else:
            mask_stack = np.asarray(mask_data)

        n_total = len(mask_stack)
        for k, mask_prob in enumerate(mask_stack):
            if mask_prob is None:
                continue

            mask_prob = np.asarray(mask_prob, dtype=np.float32)
            if mask_prob.ndim != 2:
                continue

            if mask_prob.shape != (h, w):
                mask_prob = cv2.resize(mask_prob, (w, h), interpolation=cv2.INTER_LINEAR)

            # Soft mask edges for smoother visualization.
            mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=SMOOTH_SIGMA, sigmaY=SMOOTH_SIGMA)
            mask_alpha = np.clip(mask_prob, 0.0, 1.0) * MASK_ALPHA
            if float(mask_alpha.max()) < 0.01:
                continue

            color = color_for_idx(k, n_total=n_total)
            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            binary = (mask_prob >= 0.5).astype(np.uint8)
            if binary.any():
                contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                if contours:
                    cv2.drawContours(overlay, contours, -1, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                n_masks += 1

    elif result.masks is not None and getattr(result.masks, "xy", None) is not None:
        # Fallback to polygon mode if data masks are unavailable.
        polys = result.masks.xy
        n_total = len(polys)
        for k, poly in enumerate(polys):
            if poly is None or len(poly) < 3:
                continue

            pts = np.round(poly).astype(np.int32)
            color = color_for_idx(k, n_total=n_total)

            mask = np.zeros((h, w), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            mask_alpha = (mask.astype(np.float32) / 255.0) * MASK_ALPHA

            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            cv2.polylines(overlay, [pts], True, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
            n_masks += 1

    out = np.clip(overlay, 0, 255).astype(np.uint8)
    return out, n_masks


def id_from_name(path: Path) -> str:
    m = re.search(r"(\d+)(?!.*\d)", path.stem)
    return m.group(1) if m else path.stem


weights_info = collect_trained_best_weights()
preview_images = resolve_preview_images()

print("Selected images:")
for p in preview_images:
    print(f"- {p}")

print("Models for preview:")
for tag, run_name, w in weights_info:
    print(f"- {tag}: {w} (run: {run_name})")

id_tag = "_".join(id_from_name(p) for p in preview_images)

for tag, run_name, weights_path in weights_info:
    print(f"\nPreviewing {tag} from: {weights_path}")
    model = YOLO(str(weights_path))
    results = model.predict(
        source=[str(p) for p in preview_images],
        conf=PRED_CONF,
        iou=PRED_IOU,
        save=False,
        retina_masks=True,
        verbose=False,
    )

    n = len(preview_images)
    rows = int(np.ceil(n / COLS))
    fig, axes = plt.subplots(rows, COLS, figsize=(6 * COLS, 4.5 * rows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, img_path, res in zip(axes, preview_images, results):
        out, n_masks = render_masks_only(img_path, res)
        if out is None:
            ax.set_title(f"{img_path.name} (read error)", fontsize=9)
            continue
        ax.imshow(out)
        ax.set_title(f"{img_path.name} | masks: {n_masks}", fontsize=9)
        ax.axis("off")

    fig.suptitle(f"{tag} | {run_name}", fontsize=14)
    plt.tight_layout()

    if SAVE_PREVIEW_PNG:
        out_png = PREVIEW_SAVE_DIR / f"{tag}_{run_name}_ids_{id_tag}_preview_v2.png"
        fig.savefig(out_png, dpi=180, bbox_inches="tight")
        print(f"Saved preview: {out_png}")

    plt.show()

    # Free memory between models
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()






In [ ]:
# Summary table: all metrics for YOLO models from MLflow
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "colony_yolo_seg"
MODEL_ORDER = [
    "yolo26n-seg.pt",
    "yolo26s-seg.pt",
    "yolo26m-seg.pt",
    "yolo26l-seg.pt",
    "yolo26x-seg.pt",
]
PREFER_FINISHED = True  # if True, pick latest FINISHED run per model; fallback to latest any status


mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
exp = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
if exp is None:
    raise FileNotFoundError(f"MLflow experiment not found: {MLFLOW_EXPERIMENT}")

runs = client.search_runs(
    [exp.experiment_id],
    run_view_type=ViewType.ALL,
    max_results=1000,
    order_by=["attributes.start_time DESC"],
)


def extract_model_key(run):
    model_param = str(run.data.params.get("model", "")).lower()
    run_name = str(run.data.tags.get("mlflow.runName", "")).lower()

    m = re.search(r"(yolo26[nsmlx]-seg\.pt)", model_param)
    if m:
        return m.group(1)

    m = re.search(r"(yolo26[nsmlx]-seg)", run_name)
    if m:
        return m.group(1) + ".pt"

    return None


by_model_any = {}
by_model_finished = {}
for run in runs:
    key = extract_model_key(run)
    if key not in MODEL_ORDER:
        continue
    if key not in by_model_any:
        by_model_any[key] = run
    if run.info.status == "FINISHED" and key not in by_model_finished:
        by_model_finished[key] = run

selected = {}
for key in MODEL_ORDER:
    if PREFER_FINISHED and key in by_model_finished:
        selected[key] = by_model_finished[key]
    elif key in by_model_any:
        selected[key] = by_model_any[key]

if not selected:
    raise RuntimeError("No matching YOLO runs found in MLflow.")

metric_keys = sorted({k for run in selected.values() for k in run.data.metrics.keys()})
rows = []
now_ms = int(datetime.now(timezone.utc).timestamp() * 1000)

for key in MODEL_ORDER:
    run = selected.get(key)
    if run is None:
        rows.append({"model": key, "status": "NOT_FOUND"})
        continue

    start_ms = run.info.start_time or np.nan
    end_ms = run.info.end_time if run.info.end_time is not None else now_ms

    row = {
        "model": key,
        "run_name": run.data.tags.get("mlflow.runName", ""),
        "status": run.info.status,
        "run_id": run.info.run_id,
        "start_time": pd.to_datetime(start_ms, unit="ms", utc=True) if pd.notna(start_ms) else pd.NaT,
        "duration_min": (end_ms - start_ms) / 60000.0 if pd.notna(start_ms) else np.nan,
        "error_message": run.data.tags.get("error_message", ""),
    }

    for mk in metric_keys:
        row[mk] = run.data.metrics.get(mk, np.nan)

    rows.append(row)

summary_df = pd.DataFrame(rows)

base_cols = ["model", "run_name", "status", "duration_min", "run_id", "start_time", "error_message"]
metric_cols = [c for c in summary_df.columns if c not in base_cols]
summary_df = summary_df[base_cols + metric_cols]

# Optional rounded view for readability
display_df = summary_df.copy()
for c in display_df.columns:
    if pd.api.types.is_numeric_dtype(display_df[c]):
        display_df[c] = display_df[c].round(6)

display(display_df)

out_csv = Path("mlflow_models_metrics_summary_latest.csv")
summary_df.to_csv(out_csv, index=False, encoding="utf-8")
print("Saved:", out_csv.resolve())


In [ ]:
# Recalculate custom metrics for yolo26x-seg.pt without retraining (Dice/MAE/RMSE/MAPE)
from pathlib import Path
import json
import numpy as np
import pandas as pd
import cv2
import yaml
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
from ultralytics import YOLO

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "colony_yolo_seg"
X_MODEL_KEY = "yolo26x-seg.pt"
LOG_RECALC_TO_MLFLOW = True  # set False if you only need local files/print

PRED_CONF = 0.25
PRED_IOU = 0.7
PRED_MAX_DET = 300
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def resolve_data_yaml_local():
    if "DATA_YAML" in globals() and DATA_YAML is not None:
        p = Path(DATA_YAML)
        if p.exists():
            return p

    for c in [
        Path("cropped_736_aug_leaky/dataset/data.yaml"),
        Path("C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml"),
        Path("cropped_736/dataset/data.yaml"),
        Path("C:/ColonyNet/cropped_736/dataset/data.yaml"),
    ]:
        if c.exists():
            return c

    found = sorted(Path(".").rglob("cropped_736_aug_leaky/dataset/data.yaml"))
    if not found:
        found = sorted(Path(".").rglob("cropped_736/dataset/data.yaml"))
    if found:
        return found[0]

    raise FileNotFoundError("data.yaml not found")


def resolve_latest_x_run_and_weights():
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    client = MlflowClient()
    exp = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
    if exp is None:
        raise FileNotFoundError(f"Experiment not found: {MLFLOW_EXPERIMENT}")

    runs = client.search_runs(
        [exp.experiment_id],
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=1000,
        order_by=["attributes.start_time DESC"],
    )

    selected = None
    for r in runs:
        model_param = str(r.data.params.get("model", "")).lower()
        run_name = str(r.data.tags.get("mlflow.runName", "")).lower()
        if X_MODEL_KEY in model_param or "yolo26x-seg" in run_name:
            selected = r
            break

    if selected is None:
        raise RuntimeError("No MLflow run found for yolo26x-seg")

    candidates = []
    train_dir_param = selected.data.params.get("ultralytics_train_dir")
    if train_dir_param:
        candidates.append(Path(train_dir_param) / "weights" / "best.pt")

    for base in [
        Path("runs/segment/runs/colony_seg_mlflow"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
    ]:
        if not base.exists():
            continue
        for d in sorted(base.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
            if not d.is_dir() or d.name.endswith("_test"):
                continue
            if "yolo26x-seg" not in d.name.lower():
                continue
            candidates.append(d / "weights" / "best.pt")

    for c in candidates:
        if c.exists():
            return selected, c

    raise FileNotFoundError("best.pt for yolo26x-seg not found")


def resolve_test_split_dirs(data_yaml_path: Path):
    data = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8")) or {}
    test_raw = data.get("test")
    if not test_raw:
        raise KeyError("`test` path is missing in data.yaml")

    test_img_dir = Path(str(test_raw).strip().strip('"').strip("'"))
    if not test_img_dir.is_absolute():
        test_img_dir = (data_yaml_path.parent / test_img_dir).resolve()

    split_name = test_img_dir.name
    label_candidates = [data_yaml_path.parent / "labels" / split_name]
    if test_img_dir.parent.name == "images":
        label_candidates.append(test_img_dir.parent.parent / "labels" / split_name)
    label_candidates.append(Path(str(test_img_dir).replace("\\images\\", "\\labels\\").replace("/images/", "/labels/")))

    for cand in label_candidates:
        if cand.exists():
            return test_img_dir, cand

    return test_img_dir, label_candidates[0]


def read_yolo_seg_polygons(label_path: Path):
    if not label_path.exists():
        return []

    txt = label_path.read_text(encoding="utf-8", errors="ignore")
    if not txt.strip():
        return []

    txt = txt.replace("\r", "").replace("\\n", "\n")

    polygons = []
    for raw_line in txt.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 7:
            continue

        coords = []
        for token in parts[1:]:
            try:
                coords.append(float(token))
            except Exception:
                for sub in token.replace(",", " ").replace(";", " ").split():
                    try:
                        coords.append(float(sub))
                    except Exception:
                        pass

        if len(coords) < 6:
            continue
        if len(coords) % 2 == 1:
            coords = coords[:-1]

        pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
        pts = np.clip(pts, 0.0, 1.0)
        if pts.shape[0] >= 3:
            polygons.append(pts)

    return polygons


def polygons_norm_to_mask(polygons_norm, h: int, w: int):
    mask = np.zeros((h, w), dtype=np.uint8)
    if h <= 0 or w <= 0:
        return mask

    for pts in polygons_norm:
        arr = np.asarray(pts, dtype=np.float32)
        if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
            continue

        arr_px = np.empty_like(arr)
        arr_px[:, 0] = np.clip(arr[:, 0] * (w - 1), 0, w - 1)
        arr_px[:, 1] = np.clip(arr[:, 1] * (h - 1), 0, h - 1)
        cv2.fillPoly(mask, [np.round(arr_px).astype(np.int32)], 1)

    return mask


def result_to_pred_mask(result, h: int, w: int):
    mask = np.zeros((h, w), dtype=np.uint8)
    pred_count = 0

    masks_obj = getattr(result, "masks", None)
    if masks_obj is None:
        return mask, pred_count

    polys = getattr(masks_obj, "xy", None)
    if polys is not None and len(polys) > 0:
        pred_count = int(len(polys))
        for poly in polys:
            arr = np.asarray(poly, dtype=np.float32)
            if arr.ndim != 2 or arr.shape[1] != 2 or arr.shape[0] < 3:
                continue
            arr[:, 0] = np.clip(arr[:, 0], 0, w - 1)
            arr[:, 1] = np.clip(arr[:, 1], 0, h - 1)
            cv2.fillPoly(mask, [np.round(arr).astype(np.int32)], 1)
        return mask, pred_count

    data = getattr(masks_obj, "data", None)
    if data is None:
        return mask, pred_count

    arr = data.detach().cpu().numpy()
    pred_count = int(arr.shape[0])
    if arr.size == 0:
        return mask, pred_count

    union = (arr > 0.5).any(axis=0).astype(np.uint8)
    if union.shape != (h, w):
        union = cv2.resize(union, (w, h), interpolation=cv2.INTER_NEAREST)
    return union.astype(np.uint8), pred_count


def dice_score(pred_mask, gt_mask, eps: float = 1e-7):
    a = pred_mask.astype(bool)
    b = gt_mask.astype(bool)
    sa = float(a.sum(dtype=np.float64))
    sb = float(b.sum(dtype=np.float64))
    if sa == 0.0 and sb == 0.0:
        return 1.0
    inter = float(np.logical_and(a, b).sum(dtype=np.float64))
    return float((2.0 * inter + eps) / (sa + sb + eps))


def compute_custom_metrics(best_model, data_yaml: Path, imgsz: int = 736):
    test_img_dir, test_lbl_dir = resolve_test_split_dirs(data_yaml)
    test_images = [p for p in sorted(test_img_dir.iterdir()) if p.is_file() and p.suffix.lower() in IMG_EXTS]
    if not test_images:
        raise RuntimeError(f"No test images in: {test_img_dir}")

    pred_iter = best_model.predict(
        source=str(test_img_dir),
        imgsz=imgsz,
        conf=PRED_CONF,
        iou=PRED_IOU,
        max_det=PRED_MAX_DET,
        stream=True,
        verbose=False,
        save=False,
    )

    rows = []
    for res in pred_iter:
        image_path = Path(res.path)
        h, w = map(int, res.orig_shape)

        label_path = test_lbl_dir / f"{image_path.stem}.txt"
        gt_polys = read_yolo_seg_polygons(label_path)
        gt_mask = polygons_norm_to_mask(gt_polys, h, w)

        pred_mask, pred_count = result_to_pred_mask(res, h, w)
        gt_count = int(len(gt_polys))
        count_error = int(pred_count - gt_count)
        abs_error = abs(count_error)
        ape = (abs_error / gt_count) if gt_count > 0 else np.nan

        rows.append({
            "image": image_path.name,
            "label_exists": int(label_path.exists()),
            "gt_count": gt_count,
            "pred_count": int(pred_count),
            "count_error": count_error,
            "count_abs_error": abs_error,
            "count_ape": float(ape),
            "dice": dice_score(pred_mask, gt_mask),
        })

    df = pd.DataFrame(rows)
    sq = np.square(df["count_error"].to_numpy(dtype=np.float64))
    ape_valid = df["count_ape"].dropna()

    metrics = {
        "dice_M_mean": float(df["dice"].mean()),
        "dice_M_median": float(df["dice"].median()),
        "mae_count": float(df["count_abs_error"].mean()),
        "rmse_count": float(np.sqrt(sq.mean())),
        "mape_count_nonzero": float(ape_valid.mean() * 100.0) if len(ape_valid) else np.nan,
        "test_images_eval": int(len(df)),
        "test_images_missing_labels": int((df["label_exists"] == 0).sum()),
    }
    return metrics, df, test_img_dir


# ---- run ----
data_yaml = resolve_data_yaml_local()
run_x, best_weights_x = resolve_latest_x_run_and_weights()
print("Using DATA_YAML:", data_yaml.resolve())
print("Using X run:", run_x.data.tags.get("mlflow.runName"), run_x.info.run_id)
print("Using weights:", best_weights_x)

best_model_x = YOLO(str(best_weights_x))
custom_metrics, per_image_df, used_test_dir = compute_custom_metrics(best_model_x, data_yaml, imgsz=736)

print("\nCustom metrics (X, recalculated):")
for k, v in custom_metrics.items():
    print(f"- {k}: {v}")

try:
    display(per_image_df.head(10))
except Exception:
    print(per_image_df.head(10))

# Save local artifacts
out_dir = Path("runs/segment/runs/colony_seg_mlflow/x_metrics_recalc")
out_dir.mkdir(parents=True, exist_ok=True)
out_json = out_dir / "x_custom_metrics_recalc.json"
out_csv = out_dir / "x_per_image_metrics_recalc.csv"
out_json.write_text(json.dumps(custom_metrics, indent=2), encoding="utf-8")
per_image_df.to_csv(out_csv, index=False, encoding="utf-8")
print("Saved:", out_json.resolve())
print("Saved:", out_csv.resolve())

# Optionally log to MLflow as separate run (no retraining)
if LOG_RECALC_TO_MLFLOW:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    recalc_name = f"{run_x.data.tags.get('mlflow.runName','yolo26x')}_custom_metrics_recalc"
    with mlflow.start_run(run_name=recalc_name):
        mlflow.set_tags({
            "task": "segment_custom_metrics_recalc",
            "source_run_id": run_x.info.run_id,
            "source_model": X_MODEL_KEY,
            "retrain": "false",
        })
        mlflow.log_params({
            "data_yaml": str(data_yaml),
            "weights": str(best_weights_x),
            "test_dir": str(used_test_dir),
            "pred_conf": PRED_CONF,
            "pred_iou": PRED_IOU,
            "pred_max_det": PRED_MAX_DET,
        })
        mlflow.log_metrics({k: float(v) for k, v in custom_metrics.items() if pd.notna(v)})
        mlflow.log_artifact(str(out_json), artifact_path="recalc")
        mlflow.log_artifact(str(out_csv), artifact_path="recalc")

    print("Logged recalculated metrics to MLflow as separate run.")



In [ ]:
# Predict on full cropped_736/images/train using latest yolo26x-seg best weights
from pathlib import Path
from datetime import datetime
import colorsys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SOURCE_DIR = Path("cropped_736/images/train")
TARGET_PREVIEW_FILES = [
    "1127763_IMG_4363.jpg",
    "1127763_IMG_4615.jpg",
    "1127763_IMG_4672.jpg",
    "1127763_IMG_6137.jpg",
]
PRED_CONF = 0.25
PRED_IOU = 0.6
PRED_IMGSZ = 736
MASK_ALPHA = 0.55
CONTOUR_THICKNESS = 2
SMOOTH_SIGMA = 0.9


def resolve_x_best_weights_for_predict():
    candidates = []

    # from previous cells
    for g in ["best_weights_x", "best_weights"]:
        if g in globals():
            try:
                p = Path(globals()[g])
                if p.exists() and p.name == "best.pt":
                    candidates.append(p)
            except Exception:
                pass

    # latest x run directory
    for base in [
        Path("runs/segment/runs/colony_seg_mlflow"),
        Path("C:/ColonyNet/runs/segment/runs/colony_seg_mlflow"),
    ]:
        if not base.exists():
            continue
        for d in sorted(base.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
            if not d.is_dir() or d.name.endswith("_test"):
                continue
            if "yolo26x-seg" not in d.name.lower():
                continue
            p = d / "weights" / "best.pt"
            if p.exists():
                candidates.append(p)

    if not candidates:
        raise FileNotFoundError("Could not find best.pt for yolo26x-seg")

    # newest first
    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def color_for_idx(i, n_total=1):
    # High-contrast deterministic instance colors.
    if n_total < 1:
        n_total = 1
    t = ((i * 37) % n_total) / max(1, n_total - 1)
    hue = (0.02 + 0.96 * t) % 1.0
    sat = 0.95
    val = 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
    return np.array([255.0 * r, 255.0 * g, 255.0 * b], dtype=np.float32)


def render_masks_only(image_path, result):
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        return None, 0

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    overlay = rgb.copy()
    h, w = overlay.shape[:2]

    n_masks = 0
    if result.masks is not None and getattr(result.masks, "data", None) is not None:
        mask_data = result.masks.data
        if hasattr(mask_data, "detach"):
            mask_stack = mask_data.detach().cpu().numpy()
        else:
            mask_stack = np.asarray(mask_data)

        n_total = len(mask_stack)
        for k, mask_prob in enumerate(mask_stack):
            if mask_prob is None:
                continue

            mask_prob = np.asarray(mask_prob, dtype=np.float32)
            if mask_prob.ndim != 2:
                continue

            if mask_prob.shape != (h, w):
                mask_prob = cv2.resize(mask_prob, (w, h), interpolation=cv2.INTER_LINEAR)

            # Soft mask edges for smoother visualization.
            mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=SMOOTH_SIGMA, sigmaY=SMOOTH_SIGMA)
            mask_alpha = np.clip(mask_prob, 0.0, 1.0) * MASK_ALPHA
            if float(mask_alpha.max()) < 0.01:
                continue

            color = color_for_idx(k, n_total=n_total)
            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            binary = (mask_prob >= 0.5).astype(np.uint8)
            if binary.any():
                contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                if contours:
                    cv2.drawContours(overlay, contours, -1, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                n_masks += 1

    elif result.masks is not None and getattr(result.masks, "xy", None) is not None:
        # Fallback to polygon mode if data masks are unavailable.
        polys = result.masks.xy
        n_total = len(polys)
        for k, poly in enumerate(polys):
            if poly is None or len(poly) < 3:
                continue

            pts = np.round(poly).astype(np.int32)
            color = color_for_idx(k, n_total=n_total)

            mask = np.zeros((h, w), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            mask_alpha = (mask.astype(np.float32) / 255.0) * MASK_ALPHA

            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            cv2.polylines(overlay, [pts], True, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
            n_masks += 1

    out = np.clip(overlay, 0, 255).astype(np.uint8)
    return out, n_masks


if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source folder not found: {SOURCE_DIR}")

images = sorted([p for p in SOURCE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
if not images:
    raise RuntimeError(f"No images found in: {SOURCE_DIR}")

weights_path = resolve_x_best_weights_for_predict()
print(f"Using weights: {weights_path}")
print(f"Source folder: {SOURCE_DIR.resolve()}")
print(f"Images count: {len(images)}")

pred_project = Path("runs/colony_seg_mlflow")
pred_name = f"x_predict_train_{datetime.now():%Y%m%d_%H%M%S}"

model = YOLO(str(weights_path))
results = model.predict(
    source=str(SOURCE_DIR),
    conf=PRED_CONF,
    iou=PRED_IOU,
    imgsz=PRED_IMGSZ,
    save=True,
    show_boxes=False,
    retina_masks=True,
    project=str(pred_project),
    name=pred_name,
    exist_ok=True,
    verbose=True,
)

# Resolve real save directory from Ultralytics output (robust to internal path prefixes)
pred_dir = None
if results:
    try:
        sd = getattr(results[0], "save_dir", None)
        if sd:
            pred_dir = Path(sd)
    except Exception:
        pass

if pred_dir is None or not pred_dir.exists():
    # fallback search by folder name
    matches = sorted(Path("runs").rglob(pred_name), key=lambda x: x.stat().st_mtime, reverse=True)
    if matches:
        pred_dir = matches[0]

if pred_dir is None or not pred_dir.exists():
    raise FileNotFoundError(f"Prediction output folder not found for: {pred_name}")

print(f"Saved predictions to: {pred_dir.resolve()}")

# Preview ONLY requested files with custom per-instance colors
preview_paths = [SOURCE_DIR / fn for fn in TARGET_PREVIEW_FILES]
missing = [str(p) for p in preview_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing preview files:" + "\n" + "\n".join(missing))

print("Preview files:")
for p in preview_paths:
    print(f"- {p}")

preview_results = model.predict(
    source=[str(p) for p in preview_paths],
    conf=PRED_CONF,
    iou=PRED_IOU,
    imgsz=PRED_IMGSZ,
    save=False,
    show_boxes=False,
    retina_masks=True,
    verbose=False,
)

cols = 2
rows = int(np.ceil(len(preview_paths) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.8 * rows))
axes = np.array(axes).reshape(-1)
for ax in axes:
    ax.axis("off")

for ax, img_path, res in zip(axes, preview_paths, preview_results):
    out, n_masks = render_masks_only(img_path, res)
    if out is None:
        ax.set_title(f"{img_path.name} (read error)", fontsize=9)
        continue
    ax.imshow(out)
    ax.set_title(f"{img_path.name} | masks: {n_masks}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

# Save this custom preview panel
custom_png = pred_dir / "x_custom_preview_4363_4615_4672_6137_masks_only.png"
fig.savefig(custom_png, dpi=180, bbox_inches="tight")
print(f"Saved custom preview: {custom_png.resolve()}")




In [ ]:
# Predict + preview for all cropped736 models
from pathlib import Path
from datetime import datetime
import colorsys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SOURCE_DIR = Path("cropped_736/images/train")
TARGET_PREVIEW_FILES = [
    "1127763_IMG_4363.jpg",
    "1127763_IMG_4615.jpg",
    "1127763_IMG_4672.jpg",
    "1127763_IMG_6137.jpg",
]

RUN_ROOT_CANDIDATES = [
    Path("runs/segment/runs/segment/runs/colony_seg_mlflow_736"),
    Path("runs/segment/runs/segment/runs/colony_seg_mlflow"),
    Path("C:/ColonyNet/runs/segment/runs/segment/runs/colony_seg_mlflow_736"),
    Path("C:/ColonyNet/runs/segment/runs/segment/runs/colony_seg_mlflow"),
]

PRED_CONF = 0.25
PRED_IOU = 0.6
PRED_IMGSZ = 736
MASK_ALPHA = 0.55
CONTOUR_THICKNESS = 2
SMOOTH_SIGMA = 0.9
PRED_PROJECT = Path("runs/segment/runs/colony_seg_predicts_736")


def find_model_runs():
    model_dirs = []
    seen = set()

    for root in RUN_ROOT_CANDIDATES:
        if not root.exists():
            continue

        for d in sorted(root.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
            if not d.is_dir():
                continue
            name = d.name.lower()
            if name.endswith("_test"):
                continue
            if "yolo26" not in name or "-seg" not in name:
                continue
            if "cropped736" not in name:
                continue

            best_pt = d / "weights" / "best.pt"
            if not best_pt.exists():
                continue

            key = str(best_pt.resolve()).lower()
            if key in seen:
                continue
            seen.add(key)

            model_dirs.append(d)

    if not model_dirs:
        raise FileNotFoundError(
            "No model folders with cropped736 and weights/best.pt were found in:\n"
            + "\n".join(str(p.resolve()) for p in RUN_ROOT_CANDIDATES if p.exists())
        )

    model_dirs = sorted(model_dirs, key=lambda x: x.name.lower())
    return model_dirs


def color_for_idx(i, n_total=1):
    if n_total < 1:
        n_total = 1
    t = ((i * 37) % n_total) / max(1, n_total - 1)
    hue = (0.02 + 0.96 * t) % 1.0
    sat = 0.95
    val = 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
    return np.array([255.0 * r, 255.0 * g, 255.0 * b], dtype=np.float32)


def render_masks_only(image_path, result):
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        return None, 0

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    overlay = rgb.copy()
    h, w = overlay.shape[:2]
    n_masks = 0

    if result.masks is not None and getattr(result.masks, "data", None) is not None:
        mask_data = result.masks.data
        mask_stack = mask_data.detach().cpu().numpy() if hasattr(mask_data, "detach") else np.asarray(mask_data)
        n_total = len(mask_stack)

        for k, mask_prob in enumerate(mask_stack):
            if mask_prob is None:
                continue

            mask_prob = np.asarray(mask_prob, dtype=np.float32)
            if mask_prob.ndim != 2:
                continue

            if mask_prob.shape != (h, w):
                mask_prob = cv2.resize(mask_prob, (w, h), interpolation=cv2.INTER_LINEAR)

            mask_prob = cv2.GaussianBlur(mask_prob, (0, 0), sigmaX=SMOOTH_SIGMA, sigmaY=SMOOTH_SIGMA)
            mask_alpha = np.clip(mask_prob, 0.0, 1.0) * MASK_ALPHA
            if float(mask_alpha.max()) < 0.01:
                continue

            color = color_for_idx(k, n_total=n_total)
            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            binary = (mask_prob >= 0.5).astype(np.uint8)
            if binary.any():
                contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
                if contours:
                    cv2.drawContours(overlay, contours, -1, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
                n_masks += 1

    elif result.masks is not None and getattr(result.masks, "xy", None) is not None:
        polys = result.masks.xy
        n_total = len(polys)

        for k, poly in enumerate(polys):
            if poly is None or len(poly) < 3:
                continue

            pts = np.round(poly).astype(np.int32)
            color = color_for_idx(k, n_total=n_total)

            mask = np.zeros((h, w), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            mask_alpha = (mask.astype(np.float32) / 255.0) * MASK_ALPHA

            for ch in range(3):
                overlay[..., ch] = overlay[..., ch] * (1.0 - mask_alpha) + color[ch] * mask_alpha

            cv2.polylines(overlay, [pts], True, color.tolist(), CONTOUR_THICKNESS, lineType=cv2.LINE_AA)
            n_masks += 1

    out = np.clip(overlay, 0, 255).astype(np.uint8)
    return out, n_masks


if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source folder not found: {SOURCE_DIR}")

all_images = sorted([p for p in SOURCE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
if not all_images:
    raise RuntimeError(f"No images found in: {SOURCE_DIR}")

preview_paths = [SOURCE_DIR / fn for fn in TARGET_PREVIEW_FILES]
missing = [str(p) for p in preview_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing preview files:\n" + "\n".join(missing))

model_dirs = find_model_runs()
print("Found models:")
for d in model_dirs:
    print("-", d)

PRED_PROJECT.mkdir(parents=True, exist_ok=True)
predict_rows = []

for model_dir in model_dirs:
    weights_path = model_dir / "weights" / "best.pt"
    model_name = model_dir.name
    pred_name = f"{model_name}_predict_train_{datetime.now():%Y%m%d_%H%M%S}"

    print(f"\n=== Predicting with {model_name} ===")
    model = YOLO(str(weights_path))

    results = model.predict(
        source=str(SOURCE_DIR),
        conf=PRED_CONF,
        iou=PRED_IOU,
        imgsz=PRED_IMGSZ,
        save=True,
        show_boxes=False,
        retina_masks=True,
        project=str(PRED_PROJECT),
        name=pred_name,
        exist_ok=True,
        verbose=True,
    )

    pred_dir = None
    if results:
        try:
            sd = getattr(results[0], "save_dir", None)
            if sd:
                pred_dir = Path(sd)
        except Exception:
            pass

    if pred_dir is None or not pred_dir.exists():
        matches = sorted(PRED_PROJECT.rglob(pred_name), key=lambda x: x.stat().st_mtime, reverse=True)
        if matches:
            pred_dir = matches[0]

    if pred_dir is None or not pred_dir.exists():
        raise FileNotFoundError(f"Prediction output folder not found for: {pred_name}")

    preview_results = model.predict(
        source=[str(p) for p in preview_paths],
        conf=PRED_CONF,
        iou=PRED_IOU,
        imgsz=PRED_IMGSZ,
        save=False,
        show_boxes=False,
        retina_masks=True,
        verbose=False,
    )

    cols = 2
    rows = int(np.ceil(len(preview_paths) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.8 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")

    preview_mask_counts = {}
    for ax, img_path, res in zip(axes, preview_paths, preview_results):
        out, n_masks = render_masks_only(img_path, res)
        preview_mask_counts[img_path.name] = n_masks
        if out is None:
            ax.set_title(f"{img_path.name} (read error)", fontsize=9)
            continue
        ax.imshow(out)
        ax.set_title(f"{img_path.name} | masks: {n_masks}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    custom_png = pred_dir / f"{model_name}_preview_4363_4615_4672_6137_masks_only.png"
    fig.savefig(custom_png, dpi=180, bbox_inches="tight")
    plt.close(fig)

    predict_rows.append(
        {
            "model": model_name,
            "weights": str(weights_path.resolve()),
            "pred_dir": str(pred_dir.resolve()),
            "preview_png": str(custom_png.resolve()),
            "masks_4363": preview_mask_counts.get("1127763_IMG_4363.jpg"),
            "masks_4615": preview_mask_counts.get("1127763_IMG_4615.jpg"),
            "masks_4672": preview_mask_counts.get("1127763_IMG_4672.jpg"),
            "masks_6137": preview_mask_counts.get("1127763_IMG_6137.jpg"),
        }
    )

predict_df = pd.DataFrame(predict_rows).sort_values("model").reset_index(drop=True)
display(predict_df)

predict_summary_csv = PRED_PROJECT / "predict_summary_cropped736.csv"
predict_summary_xlsx = PRED_PROJECT / "predict_summary_cropped736.xlsx"
predict_df.to_csv(predict_summary_csv, index=False, encoding="utf-8-sig")
predict_df.to_excel(predict_summary_xlsx, index=False)

print(f"Saved summary CSV:  {predict_summary_csv.resolve()}")
print(f"Saved summary XLSX: {predict_summary_xlsx.resolve()}")


In [ ]:
# Validation summary table + comparison charts + training curves
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
import yaml

RUN_ROOT_CANDIDATES = [
    Path("runs/segment/runs/segment/runs/colony_seg_mlflow_736"),
    Path("runs/segment/runs/segment/runs/colony_seg_mlflow"),
    Path("C:/ColonyNet/runs/segment/runs/segment/runs/colony_seg_mlflow_736"),
    Path("C:/ColonyNet/runs/segment/runs/segment/runs/colony_seg_mlflow"),
]

DATA_YAML_CANDIDATES = [
    Path("cropped_736_aug_leaky/dataset/data.yaml"),
    Path("cropped_736/dataset/data.yaml"),
    Path("C:/ColonyNet/cropped_736_aug_leaky/dataset/data.yaml"),
    Path("C:/ColonyNet/cropped_736/dataset/data.yaml"),
]

VAL_IMGSZ = 736
VAL_CONF = 0.25
VAL_IOU = 0.6
VAL_BATCH = 1
OUT_DIR = Path("runs/segment/runs/colony_seg_compare_736") / f"metrics_{datetime.now():%Y%m%d_%H%M%S}"


def find_model_runs():
    model_dirs = []
    seen = set()

    for root in RUN_ROOT_CANDIDATES:
        if not root.exists():
            continue

        for d in sorted(root.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
            if not d.is_dir():
                continue
            name = d.name.lower()
            if name.endswith("_test"):
                continue
            if "yolo26" not in name or "-seg" not in name:
                continue
            if "cropped736" not in name:
                continue

            best_pt = d / "weights" / "best.pt"
            if not best_pt.exists():
                continue

            key = str(best_pt.resolve()).lower()
            if key in seen:
                continue
            seen.add(key)
            model_dirs.append(d)

    if not model_dirs:
        raise FileNotFoundError("No cropped736 model folders with weights/best.pt found.")

    return sorted(model_dirs, key=lambda x: x.name.lower())


def resolve_data_yaml():
    for p in DATA_YAML_CANDIDATES:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find data.yaml for cropped_736 dataset.")


def resolve_eval_split(data_yaml_path):
    cfg = yaml.safe_load(Path(data_yaml_path).read_text(encoding="utf-8"))
    for split in ["test", "val"]:
        split_path = cfg.get(split)
        if split_path and Path(split_path).exists():
            return split
    return "val"


def safe_metric(obj, attr_chain, default=np.nan):
    cur = obj
    try:
        for part in attr_chain.split("."):
            cur = getattr(cur, part)
        return float(cur)
    except Exception:
        return default


def resolve_results_csv(model_dir):
    candidates = [
        model_dir / "results_736_4.csv",
        model_dir / "results.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def short_model_label(name):
    name = str(name)
    for tag in ["yolo26n", "yolo26s", "yolo26m", "yolo26l", "yolo26x"]:
        if tag in name.lower():
            return tag
    return name


model_dirs = find_model_runs()
data_yaml = resolve_data_yaml()
eval_split = resolve_eval_split(data_yaml)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using DATA_YAML: {data_yaml.resolve()}")
print(f"Using split: {eval_split}")

rows = []
curve_tables = {}

for model_dir in model_dirs:
    model_name = model_dir.name
    model_label = short_model_label(model_name)
    weights_path = model_dir / "weights" / "best.pt"
    print(f"\n=== Validating {model_label} ===")

    model = YOLO(str(weights_path))
    metrics = model.val(
        data=str(data_yaml),
        split=eval_split,
        imgsz=VAL_IMGSZ,
        conf=VAL_CONF,
        iou=VAL_IOU,
        batch=VAL_BATCH,
        plots=False,
        verbose=False,
    )

    row = {
        "model": model_name,
        "model_short": model_label,
        "weights": str(weights_path.resolve()),
        "box_precision": safe_metric(metrics, "box.mp"),
        "box_recall": safe_metric(metrics, "box.mr"),
        "box_mAP50": safe_metric(metrics, "box.map50"),
        "box_mAP50_95": safe_metric(metrics, "box.map"),
        "seg_precision": safe_metric(metrics, "seg.mp"),
        "seg_recall": safe_metric(metrics, "seg.mr"),
        "seg_mAP50": safe_metric(metrics, "seg.map50"),
        "seg_mAP50_95": safe_metric(metrics, "seg.map"),
        "fitness": safe_metric(metrics, "fitness"),
        "preprocess_ms": np.nan,
        "inference_ms": np.nan,
        "postprocess_ms": np.nan,
    }

    try:
        speed = getattr(metrics, "speed", None)
        if isinstance(speed, dict):
            row["preprocess_ms"] = float(speed.get("preprocess", np.nan))
            row["inference_ms"] = float(speed.get("inference", np.nan))
            row["postprocess_ms"] = float(speed.get("postprocess", np.nan))
    except Exception:
        pass

    results_csv = resolve_results_csv(model_dir)
    row["results_csv"] = str(results_csv.resolve()) if results_csv else ""

    if results_csv is not None:
        df_curve = pd.read_csv(results_csv)
        df_curve.columns = [c.strip() for c in df_curve.columns]
        curve_tables[model_label] = df_curve

        if "epoch" in df_curve.columns:
            row["epochs_logged"] = int(df_curve["epoch"].max()) + 1
        else:
            row["epochs_logged"] = len(df_curve)

        last_row = df_curve.iloc[-1]

        for col in [
            "train/box_loss", "train/seg_loss", "train/cls_loss", "train/dfl_loss",
            "val/box_loss", "val/seg_loss", "val/cls_loss", "val/dfl_loss",
            "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)",
            "metrics/precision(M)", "metrics/recall(M)", "metrics/mAP50(M)", "metrics/mAP50-95(M)",
            "lr/pg0", "lr/pg1", "lr/pg2",
        ]:
            row[col] = float(last_row[col]) if col in df_curve.columns else np.nan
    else:
        row["epochs_logged"] = np.nan

    rows.append(row)

summary_df = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
display(summary_df)

summary_csv = OUT_DIR / "model_metrics_summary_736.csv"
summary_xlsx = OUT_DIR / "model_metrics_summary_736.xlsx"
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
summary_df.to_excel(summary_xlsx, index=False)

print(f"Saved summary CSV:  {summary_csv.resolve()}")
print(f"Saved summary XLSX: {summary_xlsx.resolve()}")

# Comparison bar charts
bar_metrics = [
    "seg_mAP50_95",
    "seg_mAP50",
    "seg_precision",
    "seg_recall",
    "box_mAP50_95",
    "inference_ms",
]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.reshape(-1)

for ax, metric_name in zip(axes, bar_metrics):
    plot_df = summary_df[["model_short", metric_name]].dropna().sort_values(metric_name, ascending=(metric_name == "inference_ms"))
    if plot_df.empty:
        ax.set_title(f"{metric_name} (no data)")
        ax.axis("off")
        continue

    ax.bar(plot_df["model_short"], plot_df[metric_name])
    ax.set_title(metric_name)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

# Training curves from results.csv / results_736_4.csv
curve_specs = [
    ("train/seg_loss", "Train Seg Loss"),
    ("val/seg_loss", "Val Seg Loss"),
    ("metrics/mAP50(M)", "Mask mAP50"),
    ("metrics/mAP50-95(M)", "Mask mAP50-95"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.reshape(-1)

for ax, (col, title) in zip(axes, curve_specs):
    plotted = False
    for model_name, df_curve in curve_tables.items():
        if col not in df_curve.columns:
            continue
        x = df_curve["epoch"] if "epoch" in df_curve.columns else np.arange(len(df_curve))
        ax.plot(x, df_curve[col], marker="o", linewidth=2, label=model_name)
        plotted = True

    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.25)
    if plotted:
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.show()

# Optional ranking by main mask metric
rank_df = summary_df.sort_values(["seg_mAP50_95", "seg_mAP50"], ascending=False).reset_index(drop=True)
display(rank_df[[
    "model",
    "seg_mAP50_95",
    "seg_mAP50",
    "seg_precision",
    "seg_recall",
    "box_mAP50_95",
    "inference_ms",
    "epochs_logged",
]])


# YOLO Colony Segmentation Metrics Report

Источник метрик: `C:\ColonyNet\mlflow_models_metrics_summary_latest.csv`

## Что означает каждая метрика

- `mAP50_95_M`: основная метрика качества сегментации масок. Чем выше, тем лучше. Это средний AP по IoU от 0.50 до 0.95.
- `mAP50_M`: более мягкая метрика качества масок при IoU = 0.50. Обычно выше, чем `mAP50_95_M`.
- `precision_M`: доля корректных масок среди предсказанных масок.
- `recall_M`: доля найденных объектов среди всех размеченных объектов.
- `mAP50_95_B`, `mAP50_B`, `precision_B`, `recall_B`: те же метрики, но для боксов, а не для масок.
- `dice_M_mean`: средний Dice по маскам. Удобен как дополнительная метрика overlap.
- `dice_M_median`: медианный Dice. Менее чувствителен к сильным выбросам.
- `mae_count`: средняя абсолютная ошибка по числу найденных колоний.
- `rmse_count`: среднеквадратичная ошибка по числу найденных колоний. Сильнее штрафует большие ошибки.
- `mape_count_nonzero`: относительная ошибка по числу объектов, рассчитанная на изображениях, где истинное число объектов не равно нулю.
- `fitness`: агрегированная служебная метрика Ultralytics для ранжирования модели. Полезна как общий ориентир, но для вашей задачи важнее смотреть прежде всего на mask-метрики.
- `duration_min`: время обучения в минутах.

## Сводка по моделям

| Model | mAP50-95 Mask | mAP50 Mask | Precision Mask | Recall Mask | Dice Mean | MAE Count | RMSE Count | MAPE Count | Duration, min |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| yolo26n-seg.pt | 0.3002 | 0.5684 | 0.8859 | 0.3932 | 0.7834 | 191.4688 | 394.5202 | 18.7775 | 37.4509 |
| yolo26s-seg.pt | 0.3836 | 0.6541 | 0.9362 | 0.4042 | 0.7886 | 191.2344 | 396.1689 | 17.7441 | 59.8995 |
| yolo26m-seg.pt | 0.4227 | 0.6713 | 0.9586 | 0.4160 | 0.7903 | 190.3125 | 394.6725 | 17.4073 | 322.1299 |
| yolo26l-seg.pt | 0.4314 | 0.6808 | 0.9650 | 0.4148 | 0.7901 | 190.4688 | 395.1548 | 17.3974 | 502.6380 |
| yolo26x-seg.pt | 0.4545 | 0.6942 | 0.9777 | 0.4201 | 0.7904 | 190.0625 | 394.4362 | 17.2524 | 1331.8361 |

## Вывод

- Лучшая модель по основной метрике сегментации `mAP50_95_M`: `yolo26x-seg.pt` (`0.4545`).
- Лучшая модель по `mAP50_M`: `yolo26x-seg.pt` (`0.6942`).
- Лучшая модель по `precision_M`: `yolo26x-seg.pt` (`0.9777`).
- Лучшая модель по `recall_M`: `yolo26x-seg.pt` (`0.4201`), но прирост recall относительно `m/l` уже небольшой.
- Лучшая модель по Dice: `yolo26x-seg.pt` (`0.7904`), но разница с `m` и `l` минимальна.
- По метрикам счета объектов `x` тоже лучшая, но выигрыш там уже небольшой.

## Практическая интерпретация

- Линейка `n -> s -> m -> l -> x` дает ожидаемый рост качества.
- Самый заметный скачок качества идет от `n/s` к `m`.
- Переход `m -> l -> x` улучшает метрики, но уже с убывающей отдачей.
- `yolo26x-seg.pt` дает лучший результат, но цена очень высокая: обучение заняло примерно `1331.8` мин, то есть около `22.2` часов.
- `yolo26m-seg.pt` выглядит наиболее разумным компромиссом между качеством и временем.

## Что смотреть в первую очередь для этой задачи

- Для сегментации колоний главный приоритет: `mAP50_95_M`.
- Затем: `mAP50_M`, `precision_M`, `recall_M`.
- `Dice` полезен как дополнительная проверка качества формы масок.
- Box-метрики вторичны, потому что ваша цель не боксы, а именно instance segmentation.

